# Homework 2 Solution

Select two regions, organize each regional file, and merge them into panel data.

This example uses Guangxi and Hebei. You can replace either file with `jilin.xls` to create another valid solution.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "homework_02"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

**Optional setup**

In [ ]:
%pip install -q xlrd

In [ ]:
import pandas as pd
import numpy as np

## Helper function

In [ ]:
def build_panel_from_province(file_path, region_cn):
    raw = pd.read_excel(
        file_path,
        sheet_name=0,
        header=3,
    ).rename(columns=lambda x: x.split("年")[0] if isinstance(x, str) else x)

    raw["地区"] = region_cn

    panel = raw.melt(
        id_vars=["地区", "指标"],
        var_name="年份",
    ).pivot_table(
        values="value",
        index=["地区", "年份"],
        columns="指标",
    ).reset_index().rename_axis("", axis=1)

    return panel

## Build panel data for Guangxi

In [ ]:
data_gx = build_panel_from_province(DATA_DIR / "china" / "guangxi.xls", "广西")
data_gx.head()

## Build panel data for Hebei

In [ ]:
data_hb = build_panel_from_province(DATA_DIR / "china" / "hebei.xls", "河北")
data_hb.head()

## Concatenate the two province-level panel datasets

In [ ]:
df = pd.concat([data_gx, data_hb], ignore_index=True)
df.head()

In [ ]:
# Check the data types.
df.dtypes

In [ ]:
# Convert Year from object to integer.
df["年份"] = df["年份"].astype(str).astype(int)
df.dtypes

In [ ]:
df.to_csv(MODULE_OUTPUT_DIR / "Homework2_panel_data.csv", index=False)